# Create and inspect the canonical merged dataset

Issue #28 concatenates the two local raw Parquet shards in fixed filename order.
All 11 columns, values, embedded image bytes, and rows are retained. The saved
artifact is the input for later preparation; **canonical here means one merged
source artifact, not normalized labels or a training-ready dataset**.

Select the repository `.venv` kernel. Start from the repository root or `notebooks/`.
Setup, helper definitions, and synthetic checks use no real data. **Run the merge
and inspection cells only after explicit owner authorization** under `AGENTS.md`.
Creation scans both raw shards and writes ignored `data/interim/` artifacts.
Reuse verifies the existing merged artifact without writing. Inspection decodes
only 10 sampled images on CPU. No downloads, models, splits, filtering, deduplication,
label normalization, or repairs are performed. Completed artifacts are verified
and reused; incomplete or corrupt artifact pairs cause an error.

**Execution status:** earlier owner-run creation/inspection evidence is recorded in
[the dataset document](../docs/dataset.md#saved-execution-evidence--2026-10-19).
The reuse path has synthetic coverage and successful owner-run saved evidence.
Owner acceptance remains separate. Committed outputs and execution counts are cleared.


## Setup and parameters

The two filenames define the source order; the original column order comes from
the first shard. `BATCH_SIZE` bounds image-bearing batches, not the number of rows
validated. ID counters cover the full dataset. `IMAGE_SAMPLE_SEED` controls the
repeatable sample. Libraries come from the existing environment; do not install
packages from this notebook.


In [ ]:
import base64
import hashlib
import json
import math
import random
import sys
from collections import Counter
from datetime import UTC, datetime
from importlib.metadata import version
from io import BytesIO
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter

import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from IPython.display import Markdown, display
from PIL import Image

working_directory = Path.cwd().resolve()
ROOT = working_directory.parent if working_directory.name == "notebooks" else working_directory
if not ((ROOT / "pyproject.toml").is_file() and (ROOT / "src/vlm_finetuning").is_dir()):
    raise FileNotFoundError("Start from the repository root or its notebooks/ directory.")

SHARD_NAMES = (
    "train-00000-of-00002-6cff4c59f91661c3.parquet",
    "train-00001-of-00002-bb459e5ac5f01e71.parquet",
)
SOURCE_COLUMNS = {
    "image",
    "id",
    "gender",
    "masterCategory",
    "subCategory",
    "articleType",
    "baseColour",
    "season",
    "year",
    "usage",
    "productDisplayName",
}
BATCH_SIZE = 512
IMAGE_SAMPLE_SEED = 42
IMAGE_SAMPLE_COUNT = 10

## Merge and validation helpers

Validate both schemas before creating an output. Write Arrow batches without a
pandas conversion, then reread the saved file and compare **every record in source
order**, including nested image bytes and nulls. NaNs compare as missing numeric
values without modifying them. Schema metadata is checked as well as field types.
Source SHA-256 checksums before/after detect changes during the operation.

Files are created exclusively, so reruns cannot overwrite either artifact. An
ordinary exception removes only incomplete artifacts created by that invocation.
An interrupted process can leave a partial file: stop and inspect it rather than
bypassing the protection. The manifest is written last and marks a completed merge.
Missing/repeated IDs are reported as failed quality checks in the manifest, while
all records remain preserved. They do not silently trigger filtering or repair.


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def id_report(counts: Counter, missing: int, rows: int) -> dict:
    return {
        "rows_checked": rows,
        "missing_ids": missing,
        "unique_non_null_ids": len(counts),
        "repeated_ids": sum(count > 1 for count in counts.values()),
        "extra_rows_sharing_ids": sum(count - 1 for count in counts.values()),
        "complete_and_unique": missing == 0 and len(counts) == rows,
    }


def values_equal(left: object, right: object) -> bool:
    # Arrow equality treats NaN != NaN. Use this fallback only for unequal batches.
    if isinstance(left, float) and isinstance(right, float):
        return left == right or (math.isnan(left) and math.isnan(right))
    if isinstance(left, dict) and isinstance(right, dict):
        return left.keys() == right.keys() and all(
            values_equal(left[key], right[key]) for key in left
        )
    if isinstance(left, list) and isinstance(right, list):
        return len(left) == len(right) and all(
            values_equal(a, b) for a, b in zip(left, right, strict=True)
        )
    return left == right


def validate_saved_records(
    sources: tuple[Path, ...], output: Path, schema: pa.Schema, expected_rows: int, batch_size: int
) -> int:
    compared = 0
    with pq.ParquetFile(output) as saved:
        if not saved.schema_arrow.equals(schema, check_metadata=True):
            raise ValueError("Saved schema differs from the source schema.")
        if saved.metadata.num_rows != expected_rows:
            raise ValueError("Saved metadata count differs from the combined source count.")
        saved_batches = iter(saved.iter_batches(batch_size=batch_size, use_threads=False))
        current = next(saved_batches, None)
        saved_offset = 0
        for source in sources:
            with pq.ParquetFile(source) as raw:
                for batch in raw.iter_batches(batch_size=batch_size, use_threads=False):
                    source_offset = 0
                    while source_offset < batch.num_rows:
                        if current is None:
                            raise ValueError(f"Saved artifact ends early at row {compared}.")
                        length = min(
                            batch.num_rows - source_offset, current.num_rows - saved_offset
                        )
                        left = batch.slice(source_offset, length)
                        right = current.slice(saved_offset, length)
                        if not left.equals(right) and not values_equal(
                            left.to_pylist(), right.to_pylist()
                        ):
                            raise ValueError(f"Record preservation failed near row {compared}.")
                        compared += length
                        source_offset += length
                        saved_offset += length
                        if saved_offset == current.num_rows:
                            current = next(saved_batches, None)
                            saved_offset = 0
        if current is not None or compared != expected_rows:
            raise ValueError("Saved artifact has extra rows or an incorrect scanned count.")
    return compared


def create_canonical_dataset(root: Path, batch_size: int = BATCH_SIZE) -> dict:
    if batch_size < 1:
        raise ValueError("batch_size must be positive.")
    root = root.resolve()
    sources = tuple(root / "data/raw" / name for name in SHARD_NAMES)
    output_dir = root / "data/interim"
    output = output_dir / "fashion_products_merged.parquet"
    manifest_path = output_dir / "fashion_products_merged.manifest.json"
    for path in (*sources, output, manifest_path):
        if not path.resolve().is_relative_to(root):
            raise ValueError(f"Path resolves outside the repository: {path}")
    for path in (output, manifest_path):
        if path.exists() or path.is_symlink():
            raise FileExistsError(f"Refusing to overwrite existing artifact: {path}")
    for path in sources:
        if not path.is_file():
            raise FileNotFoundError(f"Missing local Parquet shard: {path}")

    started = perf_counter()
    source_info = []
    schema = None
    for order, path in enumerate(sources):
        with pq.ParquetFile(path) as raw:
            candidate = raw.schema_arrow
            if len(candidate) != 11 or set(candidate.names) != SOURCE_COLUMNS:
                raise ValueError(f"Expected the 11 original source columns: {path.name}")
            if schema is None:
                schema = candidate
            elif not candidate.equals(schema, check_metadata=True):
                raise ValueError(f"Source schema mismatch: {path.name}")
            source_info.append(
                {
                    "order": order,
                    "path": path.relative_to(root).as_posix(),
                    "size_bytes": path.stat().st_size,
                    "rows": raw.metadata.num_rows,
                    "sha256": sha256_file(path),
                }
            )
    if schema is None:
        raise ValueError("No source schema found.")
    expected_rows = sum(item["rows"] for item in source_info)
    all_ids = Counter()
    total_missing = 0
    sampled_positions = set(
        random.Random(IMAGE_SAMPLE_SEED).sample(
            range(expected_rows), min(IMAGE_SAMPLE_COUNT, expected_rows)
        )
    )
    sampled_rows = []
    global_row = 0
    created = []
    output_dir.mkdir(parents=True, exist_ok=True)
    try:
        with output.open("xb") as sink:
            created.append(output)
            with pq.ParquetWriter(sink, schema, compression="snappy") as writer:
                for path, info in zip(sources, source_info, strict=True):
                    counts = Counter()
                    rows = missing = 0
                    with pq.ParquetFile(path) as raw:
                        if not raw.schema_arrow.equals(schema, check_metadata=True):
                            raise ValueError(f"Source schema changed: {path.name}")
                        for batch in raw.iter_batches(batch_size=batch_size, use_threads=False):
                            writer.write_batch(batch)
                            rows += batch.num_rows
                            for value in batch.column(schema.get_field_index("id")).to_pylist():
                                if global_row in sampled_positions:
                                    sampled_rows.append({"merged_row": global_row, "id": value})
                                global_row += 1
                                if value is None:
                                    missing += 1
                                else:
                                    counts[value] += 1
                    if rows != info["rows"]:
                        raise ValueError(f"Scanned source count mismatch: {path.name}")
                    info["ids"] = id_report(counts, missing, rows)
                    all_ids.update(counts)
                    total_missing += missing
        compared = validate_saved_records(sources, output, schema, expected_rows, batch_size)
        for path, info in zip(sources, source_info, strict=True):
            if sha256_file(path) != info["sha256"]:
                raise ValueError(f"Raw source changed during merge: {path.name}")
        ids = id_report(all_ids, total_missing, compared)
        identity = [{key: info[key] for key in ("order", "path", "sha256")} for info in source_info]
        snapshot = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
        manifest = {
            "manifest_version": 1,
            "created_at_utc": datetime.now(UTC).isoformat(),
            "source_dataset": "ashraq/fashion-product-images-small",
            "upstream_revision": None,
            "revision_status": "unresolved upstream revision; local snapshot",
            "local_snapshot_id": f"sha256:{snapshot}",
            "sources": source_info,
            "operation": "concatenate all rows in listed source order; no transformations",
            "schema": {
                "fields": [
                    {"name": field.name, "type": str(field.type), "nullable": field.nullable}
                    for field in schema
                ],
                "arrow_ipc_base64": base64.b64encode(schema.serialize().to_pybytes()).decode(),
            },
            "output": {
                "path": output.relative_to(root).as_posix(),
                "rows": compared,
                "size_bytes": output.stat().st_size,
                "sha256": sha256_file(output),
            },
            "validation": {
                "schema_matches_including_metadata": True,
                "combined_row_count_matches": compared == expected_rows,
                "records_preserved_in_order_including_image_bytes": True,
                "records_compared": compared,
                "raw_checksums_unchanged": True,
                "ids": ids,
            },
            "inspection": {
                "seed": IMAGE_SAMPLE_SEED,
                "requested_images": IMAGE_SAMPLE_COUNT,
                "sampled_rows": sampled_rows,
                "status": "pending; run saved-artifact inspection cells",
            },
            "runtime": {
                "python": sys.version.split()[0],
                "pyarrow": version("pyarrow"),
                "pandas": version("pandas"),
                "Pillow": version("Pillow"),
                "device": "cpu",
                "batch_size": batch_size,
                "merge_validation_seconds": perf_counter() - started,
            },
        }
        with manifest_path.open("x", encoding="utf-8") as stream:
            created.append(manifest_path)
            json.dump(manifest, stream, indent=2, allow_nan=False)
            stream.write("\n")
    except Exception:
        for path in reversed(created):
            path.unlink()
        raise
    return manifest


def create_or_reuse_canonical_dataset(root: Path, batch_size: int = BATCH_SIZE) -> dict:
    try:
        return create_canonical_dataset(root, batch_size)
    except FileExistsError as error:
        root = root.resolve()
        output = root / "data/interim/fashion_products_merged.parquet"
        manifest_path = root / "data/interim/fashion_products_merged.manifest.json"
        if not (output.is_file() and manifest_path.is_file()):
            raise FileExistsError(
                "Cannot reuse an incomplete artifact pair: both the merged Parquet and "
                "manifest must exist. Existing files were left unchanged."
            ) from error
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if not isinstance(manifest, dict) or manifest.get("manifest_version") != 1:
            raise ValueError("Unsupported or invalid existing merge manifest.") from error
        saved_output = manifest.get("output")
        if not isinstance(saved_output, dict) or saved_output.get("path") != (
            output.relative_to(root).as_posix()
        ):
            raise ValueError(
                "Existing manifest does not identify the expected merged file."
            ) from error
        if sha256_file(output) != saved_output.get("sha256"):
            raise ValueError(
                "Existing merged artifact checksum differs from its manifest."
            ) from error
        print("Existing artifact pair found; skipping merge without overwriting.")
        print("Output checksum matched; continuing with saved-artifact inspection.")
        return manifest

## Saved-artifact inspection helpers

Read the saved Parquet file, not a cached merged table. Missingness counts Arrow
nulls plus floating-point NaNs at the top level; empty strings and nested image
fields are not reclassified. The sample selects distinct **row positions**, not
proven unique images. Decode errors are raised with the saved row position and ID;
no substitute row is silently chosen.


In [ ]:
def inspect_saved_dataset(path: Path, batch_size: int = BATCH_SIZE) -> dict:
    counts = Counter()
    rows = missing_ids = 0
    with pq.ParquetFile(path) as saved:
        schema = saved.schema_arrow
        missing = dict.fromkeys(schema.names, 0)
        for batch in saved.iter_batches(batch_size=batch_size, use_threads=False):
            rows += batch.num_rows
            for field, column in zip(schema, batch.columns, strict=True):
                missing[field.name] += column.null_count
                if pa.types.is_floating(field.type):
                    missing[field.name] += pc.sum(pc.is_nan(column)).as_py() or 0
            for value in batch.column(schema.get_field_index("id")).to_pylist():
                if value is None:
                    missing_ids += 1
                else:
                    counts[value] += 1
        if rows != saved.metadata.num_rows:
            raise ValueError("Saved inspection count differs from Parquet metadata.")
    return {
        "schema": schema,
        "rows": rows,
        "missing": missing,
        "ids": id_report(counts, missing_ids, rows),
    }


def sample_saved_images(path: Path, count: int, seed: int, batch_size: int = BATCH_SIZE) -> list:
    with pq.ParquetFile(path) as saved:
        if count < 1 or saved.metadata.num_rows < count:
            raise ValueError(f"Need at least {count} rows for the requested image sample.")
        positions = sorted(random.Random(seed).sample(range(saved.metadata.num_rows), count))
        samples = []
        offset = 0
        for batch in saved.iter_batches(batch_size=batch_size, use_threads=False):
            for position in positions:
                if offset <= position < offset + batch.num_rows:
                    record = batch.slice(position - offset, 1).to_pylist()[0]
                    payload = (record["image"] or {}).get("bytes")
                    try:
                        if not payload:
                            raise ValueError("Missing embedded image bytes.")
                        with Image.open(BytesIO(payload)) as image:
                            image.load()
                            preview = image.copy()
                    except (OSError, ValueError) as error:
                        raise ValueError(
                            f"Image decode failed at saved row {position}, ID {record['id']}"
                        ) from error
                    samples.append((position, record, preview))
            offset += batch.num_rows
            if len(samples) == count:
                break
        if len(samples) != count:
            raise ValueError("Could not read every selected image row.")
    return samples

## Bounded synthetic checks — no real data

Use 12 tiny generated PNG records in a temporary repository-local directory. Check
round-trip values/bytes/order/schema, ID defects without row loss, cross-shard IDs,
missingness, sampled image decoding, schema mismatch before writing, corrupt/reordered
saved records, counts, existing-output protection, successful reuse, incomplete
pairs, corrupt artifacts/manifests, and cleanup after failure. These
checks establish implementation behavior, not the quality of the real snapshot.


In [ ]:
from unittest.mock import patch


def run_synthetic_checks(root: Path) -> None:
    with TemporaryDirectory(prefix="issue-28-", dir=root) as temporary:
        fixture = Path(temporary)
        raw_dir = fixture / "data/raw"
        raw_dir.mkdir(parents=True)
        buffer = BytesIO()
        with Image.new("RGB", (3, 4), color="blue") as image:
            image.save(buffer, format="PNG")
        records = [
            {
                "image": {"bytes": buffer.getvalue(), "path": None},
                "id": number,
                "gender": "Unisex",
                "masterCategory": "Apparel",
                "subCategory": "Topwear",
                "articleType": " Shirts ",
                "baseColour": "Blue",
                "season": None,
                "year": 2020.0,
                "usage": "Casual",
                "productDisplayName": f"Synthetic {number}",
            }
            for number in range(12)
        ]
        records[1]["id"] = None
        records[7]["id"] = 0
        records[2]["year"] = float("nan")
        table = pa.Table.from_pylist(records).replace_schema_metadata({b"fixture": b"synthetic"})
        sources = tuple(raw_dir / name for name in SHARD_NAMES)

        def write_sources(candidate: pa.Table = table) -> None:
            pq.write_table(candidate.slice(0, 5), sources[0])
            pq.write_table(candidate.slice(5), sources[1])

        def expect_error(error_type: type[Exception], operation) -> None:
            try:
                operation()
            except error_type:
                return
            raise AssertionError(f"Expected {error_type.__name__}")

        write_sources()
        original_hashes = [sha256_file(path) for path in sources]
        manifest = create_canonical_dataset(fixture, batch_size=3)
        output = fixture / manifest["output"]["path"]
        manifest_path = output.with_suffix(".manifest.json")
        saved = pq.read_table(output)
        assert saved.schema.equals(table.schema, check_metadata=True)
        assert values_equal(saved.to_pylist(), records)
        assert saved.num_rows == 12
        assert [sha256_file(path) for path in sources] == original_hashes
        ids = manifest["validation"]["ids"]
        assert ids["missing_ids"] == 1 and ids["repeated_ids"] == 1
        assert ids["unique_non_null_ids"] == 10 and not ids["complete_and_unique"]
        assert all(item["ids"]["repeated_ids"] == 0 for item in manifest["sources"])
        assert manifest["output"]["sha256"] == sha256_file(output)
        assert manifest["upstream_revision"] is None
        assert json.loads(manifest_path.read_text()) == manifest
        expected_positions = sorted(random.Random(42).sample(range(12), 10))
        assert manifest["inspection"]["sampled_rows"] == [
            {"merged_row": position, "id": records[position]["id"]}
            for position in expected_positions
        ]
        inspected = inspect_saved_dataset(output, batch_size=4)
        assert inspected["rows"] == 12 and inspected["ids"] == ids
        assert inspected["missing"]["season"] == 12
        assert inspected["missing"]["year"] == 1
        samples = sample_saved_images(output, 10, 42, batch_size=4)
        assert len({position for position, _, _ in samples}) == 10
        assert [position for position, _, _ in samples] == expected_positions
        assert all(image.size == (3, 4) for _, _, image in samples)
        before = (output.read_bytes(), manifest_path.read_bytes())
        assert create_or_reuse_canonical_dataset(fixture, 3) == manifest
        assert before == (output.read_bytes(), manifest_path.read_bytes())
        output.write_bytes(b"corrupt saved output")
        expect_error(ValueError, lambda: create_or_reuse_canonical_dataset(fixture, 3))
        assert output.read_bytes() == b"corrupt saved output"
        output.write_bytes(before[0])
        manifest_path.write_text("invalid JSON", encoding="utf-8")
        expect_error(json.JSONDecodeError, lambda: create_or_reuse_canonical_dataset(fixture, 3))
        assert manifest_path.read_text() == "invalid JSON"
        manifest_path.write_bytes(before[1])
        expect_error(FileExistsError, lambda: create_canonical_dataset(fixture, 3))
        assert before == (output.read_bytes(), manifest_path.read_bytes())
        output.unlink()
        expect_error(FileExistsError, lambda: create_or_reuse_canonical_dataset(fixture, 3))
        expect_error(FileExistsError, lambda: create_canonical_dataset(fixture, 3))
        assert not output.exists() and manifest_path.read_bytes() == before[1]
        manifest_path.unlink()
        output.write_bytes(b"existing output")
        expect_error(FileExistsError, lambda: create_or_reuse_canonical_dataset(fixture, 3))
        expect_error(FileExistsError, lambda: create_canonical_dataset(fixture, 3))
        assert output.read_bytes() == b"existing output" and not manifest_path.exists()
        output.unlink()

        bad = fixture / "bad.parquet"
        for damaged in (table.slice(0, 11), table.take(list(reversed(range(12))))):
            pq.write_table(damaged, bad)
            expect_error(
                ValueError, lambda: validate_saved_records(sources, bad, table.schema, 12, 3)
            )
        altered = list(records)
        altered[0] = {**records[0], "image": {"bytes": b"changed", "path": None}}
        pq.write_table(pa.Table.from_pylist(altered, schema=table.schema), bad)
        expect_error(ValueError, lambda: validate_saved_records(sources, bad, table.schema, 12, 3))
        pq.write_table(table.drop(["usage"]).slice(5), sources[1])
        expect_error(ValueError, lambda: create_canonical_dataset(fixture, 3))
        assert not output.exists() and not manifest_path.exists()
        changed_type = table.set_column(
            table.schema.get_field_index("year"), "year", pa.array(["2020"] * 12)
        )
        pq.write_table(changed_type.slice(5), sources[1])
        expect_error(ValueError, lambda: create_canonical_dataset(fixture, 3))
        assert not output.exists() and not manifest_path.exists()
        write_sources()
        with patch.dict(
            create_canonical_dataset.__globals__,
            {"validate_saved_records": lambda *args: (_ for _ in ()).throw(ValueError("injected"))},
        ):
            expect_error(ValueError, lambda: create_canonical_dataset(fixture, 3))
        assert not output.exists() and not manifest_path.exists()
        assert [sha256_file(path) for path in sources] == original_hashes
        clean = table.set_column(table.schema.get_field_index("id"), "id", pa.array(range(12)))
        write_sources(clean)
        clean_manifest = create_or_reuse_canonical_dataset(fixture, 4)
        assert clean_manifest["validation"]["ids"]["complete_and_unique"]
    print("Synthetic merge, validation, inspection, and failure-path checks passed.")


run_synthetic_checks(ROOT)

## Create or reuse the merged artifact — real data, owner authorization required

This cell reads both full raw shards several times for hashing, streaming merge,
and round-trip comparison. CPU memory holds batches plus ID counters, rather than
all decoded images. It writes the merged Parquet and JSON manifest under ignored
`data/interim/`. Reserve disk space comparable to the approximately 259 MiB source
Parquet total plus compression/metadata overhead; actual size and duration are
measured by the run, not promised. The raw files remain provenance/rebuild inputs.

The source revision is unresolved. The manifest identifies the ordered local
snapshot using its SHA-256 checksums; filename hashes are not upstream revisions.
An ID quality failure is shown explicitly and does not invalidate record preservation.


On rerun, the wrapper catches only `FileExistsError`. If both artifacts exist,
it loads the existing manifest and verifies the merged file's identity and SHA-256
checksum, then skips writing and continues to the inspection cells. Those cells
validate schema, full count, and IDs against the saved manifest. An incomplete pair,
invalid manifest, or checksum mismatch stops execution without changing either file.
The manifest remains the provenance record for its original source snapshot; reuse
does not rescan the current raw shards or rebuild the merged file.

Both creation and reuse require owner-authorized real-data execution. Reuse hashes
and scans the saved merged file and decodes ten sampled images on CPU, with no
artifact writes, model loading, or downloads.


In [ ]:
manifest = create_or_reuse_canonical_dataset(ROOT)
MERGED_PATH = ROOT / manifest["output"]["path"]
print(json.dumps(manifest, indent=2))
if not manifest["validation"]["ids"]["complete_and_unique"]:
    print("ID QUALITY CHECK FAILED: missing or repeated IDs; all records were preserved.")

## Inspect the saved schema, full count, missingness, and IDs

Scan every saved row, checking against the manifest and Parquet count. These are
full-dataset summaries; they are not extrapolated from the previews below.


In [ ]:
inspection = inspect_saved_dataset(MERGED_PATH)
expected_schema = pa.ipc.read_schema(
    pa.BufferReader(base64.b64decode(manifest["schema"]["arrow_ipc_base64"]))
)
if not inspection["schema"].equals(expected_schema, check_metadata=True):
    raise ValueError("Inspection schema differs from the merge manifest.")
if inspection["rows"] != manifest["output"]["rows"]:
    raise ValueError("Inspection row count differs from the merge manifest.")
if inspection["ids"] != manifest["validation"]["ids"]:
    raise ValueError("Saved ID summary differs from the source ID summary.")
if sha256_file(MERGED_PATH) != manifest["output"]["sha256"]:
    raise ValueError("Saved artifact checksum differs from the merge manifest.")

print(f"Column count: {len(inspection['schema'])}")
print(f"Full scanned row count: {inspection['rows']:,}")
display(
    pd.DataFrame(
        [
            {"column": field.name, "type": str(field.type), "nullable": field.nullable}
            for field in inspection["schema"]
        ]
    )
)
display(
    pd.DataFrame(
        [
            {
                "column": name,
                "missing_values": count,
                "missing_percent": 100 * count / inspection["rows"] if inspection["rows"] else 0.0,
            }
            for name, count in inspection["missing"].items()
        ]
    )
)
display(pd.DataFrame([inspection["ids"]]))
if not inspection["ids"]["complete_and_unique"]:
    print("ID QUALITY CHECK FAILED: missing or repeated IDs require later policy review.")

## First 100 raw rows from the saved artifact

Display all column headers and the first 100 rows in merged source order. Only
long text representations are shortened, including image bytes; underlying values
are retained. A smaller dataset displays its complete available rows.


In [ ]:
with pq.ParquetFile(MERGED_PATH) as saved:
    first_batch = next(saved.iter_batches(batch_size=100, use_threads=False), None)
    first_100_rows = (
        first_batch.to_pandas()
        if first_batch is not None
        else pd.DataFrame(columns=saved.schema_arrow.names)
    )
with pd.option_context(
    "display.max_rows",
    100,
    "display.min_rows",
    100,
    "display.max_columns",
    None,
    "display.max_colwidth",
    80,
):
    display(first_100_rows)

## Ten random images with original column names and raw metadata

Sample 10 distinct positions from the entire saved dataset with the configured
seed, then display them in source order. All original fields accompany each image.
`random_image_samples` retains full records and bytes; table shortening is display
only. No product names or metadata are passed to a model.


In [ ]:
random_image_samples = sample_saved_images(MERGED_PATH, IMAGE_SAMPLE_COUNT, IMAGE_SAMPLE_SEED)
print(
    f"Sample seed: {IMAGE_SAMPLE_SEED}; saved row positions: "
    f"{[position for position, _, _ in random_image_samples]}"
)
for number, (position, record, image) in enumerate(random_image_samples, start=1):
    display(Markdown(f"### Image {number} — merged row {position}, ID {record['id']}"))
    display(image)
    metadata = pd.DataFrame(
        [record], columns=inspection["schema"].names, index=pd.Index([position], name="merged_row")
    )
    with pd.option_context("display.max_columns", None, "display.max_colwidth", 80):
        display(metadata)

## Interpretation and separately planned preparation

After the authorized run, review the actual summaries, first 100 rows, and sampled
images. The manifest records merge validation and exact sampled row positions/IDs; its
inspection status deliberately
remains pending because executing display code cannot certify human visual review.
No real-data result is asserted in this cleared notebook.

Unique IDs do not prove unique images. Separately propose product/image-duplicate
grouping (including near duplicates and related variants), conflicting-label and
unreadable-image policy, reproducible group-disjoint train/validation/test preparation,
training-only vocabulary/eligibility decisions, and held-out test protection.
Do not choose split ratios or implement these steps here. The output keys remain
`masterCategory`, `subCategory`, `articleType`, and `baseColour`; the canonical merged
artifact retains all 11 raw columns.

Keep raw and interim artifacts out of Git. Before commit, clear all outputs and
execution counts, then stage only this notebook and the two issue #28 owner documents.
